In [1]:
import pandas as pd

optimized = pd.read_csv(
    "../data/processed/optimized_slot_recommendations.csv"
)

summary = pd.read_csv(
    "../data/processed/optimization_summary.csv"
)

print(optimized.columns.tolist())
print(summary)

['StockCode', 'order_frequency', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster_name', 'recommended_zone', 'distance_from_packing', 'picking_priority', 'current_distance', 'distance_saved', 'recommended_distance', 'optimized_slot', 'optimized_x', 'optimized_y', 'optimized_distance', 'actual_distance_saved']
                    Metric       Value
0  Average Before Distance  102.943593
1   Average After Distance    3.939385
2   Average Distance Saved   99.004208
3            Improvement %   96.173259
4       Products Relocated   25.000000


In [2]:
top_products = optimized.sort_values(
    "picking_priority",
    ascending=False
).head(20)

print(
    top_products[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "picking_priority",
            "actual_distance_saved"
        ]
    ]
)

    StockCode          cluster_name recommended_zone  picking_priority  \
2      85123A  High-Volume Products           Zone A           7831474   
162    85099B  High-Volume Products           Zone A           7340746   
163     22423  High-Volume Products           Zone A           7033544   
164     47566  High-Volume Products           Zone A           5865485   
165     20725  High-Volume Products           Zone A           5430550   
166     22197  High-Volume Products           Zone A           4936032   
167     84879  High-Volume Products           Zone A           4877160   
168     22720  High-Volume Products           Zone A           4754705   
169     21212  High-Volume Products           Zone A           4539480   
170     22383  High-Volume Products           Zone A           4437105   
171     22457  High-Volume Products           Zone A           4382741   
172     20727  High-Volume Products           Zone A           4377847   
173     22469  High-Volume Products   

In [3]:
top_savings = optimized.sort_values(
    "actual_distance_saved",
    ascending=False
).head(20)

print(
    top_savings[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "current_distance",
            "optimized_distance",
            "actual_distance_saved"
        ]
    ]
)

   StockCode          cluster_name recommended_zone  current_distance  \
0      23309  High-Volume Products           Zone A                60   
1      22086  High-Volume Products           Zone A                60   
5      22998  High-Volume Products           Zone A                60   
6      22616  High-Volume Products           Zone A                60   
2     85123A  High-Volume Products           Zone A                60   
10     17003  High-Volume Products           Zone A                60   
7      15036  High-Volume Products           Zone A                60   
11     71459  High-Volume Products           Zone A                60   
3      22577  High-Volume Products           Zone A                60   
15     21231  High-Volume Products           Zone A                60   
4      84836  High-Volume Products           Zone A                60   
8      22189  High-Volume Products           Zone A                60   
12     84077  High-Volume Products           Zone A

In [4]:
cluster_analysis = optimized.groupby(
    "cluster_name"
).agg(
    product_count=("StockCode", "count"),
    avg_priority=("picking_priority", "mean"),
    avg_distance_saved=("actual_distance_saved", "mean")
).sort_values(
    "avg_distance_saved",
    ascending=False
)

print(cluster_analysis)

                         product_count  avg_priority  avg_distance_saved
cluster_name                                                            
High-Volume Products               178  2.565005e+06                56.0
Active/Regular Products           2192  4.482018e+05                 NaN


In [5]:
zone_analysis = optimized.groupby(
    "recommended_zone"
).agg(
    product_count=("StockCode", "count"),
    avg_distance_saved=("actual_distance_saved", "mean")
)

print(zone_analysis)

                  product_count  avg_distance_saved
recommended_zone                                   
Zone A                      178                56.0
Zone B                     2192                 NaN


In [6]:
def make_recommendation(row):

    if row["cluster_name"] == "High-Volume Products":
        return "Place closer to packing area due to high product movement."

    elif row["cluster_name"] == "Active/Regular Products":
        return "Keep in an easily accessible zone for regular picking."

    elif row["cluster_name"] == "Low-Movement Products":
        return "Place in a moderate-distance zone to balance space and access."

    elif row["cluster_name"] == "Slow-Moving Products":
        return "Place farther from packing area because of low movement."

    return "Review product placement based on warehouse demand."

In [7]:
optimized["business_recommendation"] = optimized.apply(
    make_recommendation,
    axis=1
)

In [8]:
print(
    optimized[
        [
            "StockCode",
            "cluster_name",
            "recommended_zone",
            "business_recommendation"
        ]
    ].head(20)
)

   StockCode          cluster_name recommended_zone  \
0      23309  High-Volume Products           Zone A   
1      22086  High-Volume Products           Zone A   
2     85123A  High-Volume Products           Zone A   
3      22577  High-Volume Products           Zone A   
4      84836  High-Volume Products           Zone A   
5      22998  High-Volume Products           Zone A   
6      22616  High-Volume Products           Zone A   
7      15036  High-Volume Products           Zone A   
8      22189  High-Volume Products           Zone A   
9      22595  High-Volume Products           Zone A   
10     17003  High-Volume Products           Zone A   
11     71459  High-Volume Products           Zone A   
12     84077  High-Volume Products           Zone A   
13     20971  High-Volume Products           Zone A   
14     21094  High-Volume Products           Zone A   
15     21231  High-Volume Products           Zone A   
16     23310  High-Volume Products           Zone A   
17     206

In [9]:
business_insights = optimized[
    [
        "StockCode",
        "cluster_name",
        "recommended_zone",
        "picking_priority",
        "actual_distance_saved",
        "business_recommendation"
    ]
].sort_values(
    "picking_priority",
    ascending=False
)

In [10]:
business_insights.to_csv(
    "../data/processed/business_insights.csv",
    index=False
)

In [11]:
print("Top Priority Products:")
print(
    business_insights.head(10)[
        ["StockCode", "cluster_name", "recommended_zone"]
    ]
)

print("\nTotal Products Analyzed:", len(optimized))

print(
    "Total Estimated Distance Saved:",
    optimized["actual_distance_saved"].sum()
)

print(
    "Average Distance Saved:",
    optimized["actual_distance_saved"].mean()
)

Top Priority Products:
    StockCode          cluster_name recommended_zone
2      85123A  High-Volume Products           Zone A
162    85099B  High-Volume Products           Zone A
163     22423  High-Volume Products           Zone A
164     47566  High-Volume Products           Zone A
165     20725  High-Volume Products           Zone A
166     22197  High-Volume Products           Zone A
167     84879  High-Volume Products           Zone A
168     22720  High-Volume Products           Zone A
169     21212  High-Volume Products           Zone A
170     22383  High-Volume Products           Zone A

Total Products Analyzed: 2370
Total Estimated Distance Saved: 1400.0
Average Distance Saved: 56.0
